In [15]:
#!/usr/bin/env python
# coding: utf-8

import os
import subprocess
import concurrent.futures
from joblib import Parallel, delayed

In [36]:
    
for num_nodes in  [8]:
    
    # Fetch all existing instance names matching tsm-sc-*
    fetch_cmd = f'''
    gcloud compute instances list \
        --filter="name~'tsm-sc-'" \
        --format="value(name)"
    '''
    instances_to_delete = subprocess.check_output(fetch_cmd, shell=True).decode().strip().split('\n')
    instances_to_delete = [name for name in instances_to_delete if name]  # Remove empty entries
    
    print("\n➡ Existing instances to delete:", instances_to_delete)
    
    if instances_to_delete:
        # Use parallel deletion
        def delete_instance(instance_name):
            cmd = f'''
            gcloud compute instances delete {instance_name} \
                --zone={zone} \
                --project={project} \
                --quiet
            '''
            print(f"Deleting: {instance_name}")
            return subprocess.call(cmd, shell=True)
    
        with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
            futures = [executor.submit(delete_instance, inst) for inst in instances_to_delete]
            concurrent.futures.wait(futures)
    
        print("🧹 All old tsm-sc-* instances deleted.\n")
    else:
        print("✔ No previous instances found to delete.\n")
    
    


➡ Existing instances to delete: ['tsm-sc-000', 'tsm-sc-001', 'tsm-sc-002', 'tsm-sc-003', 'tsm-sc-004', 'tsm-sc-005', 'tsm-sc-006', 'tsm-sc-007']
Deleting: tsm-sc-000
Deleting: tsm-sc-001
Deleting: tsm-sc-002
Deleting: tsm-sc-003
Deleting: tsm-sc-004
Deleting: tsm-sc-005
Deleting: tsm-sc-006
Deleting: tsm-sc-007


Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-002].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-003].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-000].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-007].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-001].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-005].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-006].


🧹 All old tsm-sc-* instances deleted.



Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-004].


In [33]:
    
    project = "ucr-ursa-major-lesani-lab"
    zone = "us-central1-c"
    machine_type = "e2-highmem-2"
    image_family = "tsm-sc-family"  # your custom image
    subnet = "default"
    gcp_username = "tejas"
    

    # Cleanup any existing instances with same prefix
    os.system(f'gcloud compute instances delete --zone={zone} --quiet '
              f'$(gcloud compute instances list --filter="name~\'tsm-sc-\'" --format="value(name)")')
    
    # Create commands list
    commands = []
    
    for i in range(num_nodes):
        cmd = f'''
        gcloud compute instances create tsm-sc-{i:03} \
            --project={project} \
            --zone={zone} \
            --machine-type={machine_type} \
            --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
            --can-ip-forward \
            --maintenance-policy=MIGRATE \
            --provisioning-model=STANDARD \
            --service-account=961693926925-compute@developer.gserviceaccount.com \
            --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
            --tags=http-server,https-server \
            --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
            --no-shielded-secure-boot \
            --shielded-vtpm \
            --shielded-integrity-monitoring \
            --labels=goog-ec-src=vm_add-gcloud \
            --reservation-affinity=any
        '''
        commands.append(cmd.strip())
    
    
    def run_command(command):
        print(f"Running: {command}")
        return subprocess.call(command, shell=True)
    
    
    # #Parallel instance creation
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
        futures = [executor.submit(run_command, cmd) for cmd in commands]
        concurrent.futures.wait(futures)
    
    print("All instances launched.")
    
    # Wait a bit for IPs to propagate
    import time
    time.sleep(30)
    
    # Get IPs
    os.system('gcloud compute instances list --filter="name~\'tsm-sc-\'" '
              '--format="value(networkInterfaces[0].networkIP)" > tsm_ips.txt')
    
    with open('tsm_ips.txt', 'r') as f:
        iplist = [line.strip() for line in f.readlines()]
    
    print("🎯 Instance IPs:", iplist)
    

ERROR: (gcloud.compute.instances.delete) argument INSTANCE_NAMES [INSTANCE_NAMES ...]: Must be specified.
Usage: gcloud compute instances delete INSTANCE_NAMES [INSTANCE_NAMES ...] [optional flags]
  optional flags may be  --delete-disks | --help | --keep-disks | --zone

For detailed information on this command and its flags, run:
  gcloud compute instances delete --help


Running: gcloud compute instances create tsm-sc-000         --project=ucr-ursa-major-lesani-lab         --zone=us-central1-c         --machine-type=e2-highmem-2         --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default         --can-ip-forward         --maintenance-policy=MIGRATE         --provisioning-model=STANDARD         --service-account=961693926925-compute@developer.gserviceaccount.com         --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append         --tags=http-server,https-server         --create-disk=auto-delete=yes,boot=yes,image-family=tsm-sc-family,mode=rw,size=20,type=pd-balanced         --no-shielded-secure-boot         --shielded-vtpm         --shielded-integrity-monitoring         --label

Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-000].
Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-002].


NAME        ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP      STATUS
tsm-sc-000  us-central1-c  e2-highmem-2               10.128.0.53  136.115.203.213  RUNNING
NAME        ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP  STATUS
tsm-sc-002  us-central1-c  e2-highmem-2               10.128.0.55  34.59.120.4  RUNNING


Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-004].
Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-006].


...

NAME        ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP     STATUS
tsm-sc-005  us-central1-c  e2-highmem-2               10.128.0.49  34.136.195.242  RUNNING


Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-003].


NAME        ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP    STATUS
tsm-sc-003  us-central1-c  e2-highmem-2               10.128.0.54  136.113.84.71  RUNNING


Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-001].


NAME        ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP   STATUS
tsm-sc-001  us-central1-c  e2-highmem-2               10.128.0.56  34.61.28.159  RUNNING
All instances launched.
🎯 Instance IPs: ['10.128.0.53', '10.128.0.56', '10.128.0.55', '10.128.0.54', '10.128.0.38', '10.128.0.49', '10.128.0.36', '10.128.0.51']


In [35]:
    os.system('git add .; git commit -m "this works for 4 nodes "; git push')
    n_collection = 10
    
    
    

    def kill_stellar_private(i):
        remote_command = f"""\
    cd /home/tejas/stellar-private; \
    sudo pkill stellar-core; \
    """
        
        # Construct the full gcloud command
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "{remote_command}"'
        
        print(f"Executing: {command}")
    
        output = os.system(command)
        print(f"Return code for tsm-sc-{i:03}: {output}")
    
    
    results = Parallel(n_jobs=20)(delayed(kill_stellar_private)(i) for i in range(num_nodes))
    
    

    
    
    
    def git_pull_stellar(i):
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
    cd stellar-core; \
    git pull"'
        print(command)
        output = os.system(command)
        print(output)
    
    # Execute in parallel like your example
    results = Parallel(n_jobs=20)(delayed(git_pull_stellar)(i) for i in range(len(iplist)))
    print(results)
    

    import shutil
    
    if os.path.exists('../stellar-private'):
        
        shutil.rmtree('../stellar-private')
    os.mkdir('../stellar-private')
    
    
    os.system('cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh')
    
    os.system('cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; ./gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh')
    
    # --- Configuration ---
    line_to_add = "SEND_CUSTOM_MESSAGE=true"
    target_file = "../stellar-private/node1/stellar-core.cfg" 
    
    # --- The os.system() Command ---
    # This command prepends the line to the target_file on your local machine.
    os.system(f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}')
    
    print(f"The line '{line_to_add}' has been prepended to {target_file}.")
    

    def compile_stellar(i):
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
    cd stellar-core; \
    make -j16; cd; sudo rm -r stellar-private"'
        print(command)
        output = os.system(command)
        print(output)
    
    # Execute in parallel like your example
    results = Parallel(n_jobs=20)(delayed(compile_stellar)(i) for i in range(len(iplist)))
    print(results)
    
    def clean_stellar_private(i):
        remote_command = f"""\
    cd /home/tejas; \
    sudo rm -r stellar-private; \
    """
        
        # Construct the full gcloud command
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "{remote_command}"'
        
        print(f"Executing: {command}")
    
        output = os.system(command)
        print(f"Return code for tsm-sc-{i:03}: {output}")
    
    
    results = Parallel(n_jobs=20)(delayed(clean_stellar_private)(i) for i in range(num_nodes))
    
    def copy_folder_to_instance(i, source_folder = '/home/tejas/stellar-private',destination_path = '/home/tejas/stellar-private' ):
        """
        Constructs and executes the gcloud compute scp command to copy a folder
        to a specific GCP instance.
        """
        instance_name = f"tsm-sc-{i:03}"
        
        # The --recurse flag is crucial for copying folders
        # The format is: gcloud compute scp --recurse [LOCAL_SRC] [USER]@[INSTANCE_NAME]:[REMOTE_DEST]
        command = f'gcloud compute scp --zone "{zone}" --project "{project}" \
    --recurse "{source_folder}" "{instance_name}:{destination_path}"'
    
        print(f"Executing command for {instance_name}: {command}")
        
        # os.system executes the command and returns the exit status (0 for success)
        output = os.system(command)
        
        print(f"Command for {instance_name} finished with exit code: {output}")
        
        return (instance_name, output)
    
    
    results = Parallel(n_jobs=20)(
        delayed(copy_folder_to_instance)(i) for i in range(num_nodes)
    )
    
    
    
    # def setup_stellar_private(i):
    #     command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
    # cd /home/tejas; \
    # cd stellar-private; chmod +x gcp_setup_stellar_private.sh; \
    # ./gcp_setup_stellar_private.sh;"'
    #     print(command)
    #     output = os.system(command)
    #     print(output)
    
    # # Execute in parallel like your example
    # results = Parallel(n_jobs=20)(delayed(setup_stellar_private)(i) for i in range(len(iplist)))
    # print(results)
    
    def run_stellar_private(i):
        # Calculate the node number (assuming i starts at 0, node starts at 1)
        node_number = i + 1 
        instance_name = f"tsm-sc-{i:03}"
        
        # ----------------------------------------------------------------------------------
        # FIX: Use nohup, redirect I/O to a log file, and add ' & disown'
        # '2>&1' redirects stderr to stdout. '> log.txt' redirects stdout to a file.
        # '< /dev/null' ensures the process doesn't wait for input.
        # ----------------------------------------------------------------------------------
        remote_command = f"""\
    cd /home/tejas/stellar-private; \
    nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
    > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
    """
        
        # Construct the full gcloud command
        command = f'gcloud compute ssh --zone "{zone}" "{instance_name}" --project "{project}" --command "{remote_command}"'
        
        print(f"Executing: {command}")
        
        # os.system should now return immediately because the remote shell exits
        output = os.system(command)
        print(f"Return code for {instance_name}: {output}")
    
    # Corrected Loop (to run 0, 1, 2, 3)
    results = Parallel(n_jobs=20)(delayed(run_stellar_private)(i) for i in range(num_nodes))
    # results = Parallel(n_jobs=20)(delayed(run_stellar_private)(i) for i in [3,2,1,0])
    
    # time.sleep(3)
    # for i in range(num_nodes):
    
        # run_stellar_private(num_nodes-i-1)
        # time.sleep(2)
    # run_stellar_private(0)
    print(results)
    print("All SSH commands executed. Nodes should be starting up in the background.")
    
    time.sleep(190)
    
    
    
    def kill_stellar_private(i):
        remote_command = f"""\
    cd /home/tejas/stellar-private; \
    sudo pkill stellar-core; \
    """
        
        # Construct the full gcloud command
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "{remote_command}"'
        
        print(f"Executing: {command}")
    
        output = os.system(command)
        print(f"Return code for tsm-sc-{i:03}: {output}")
    
    
    results = Parallel(n_jobs=20)(delayed(kill_stellar_private)(i) for i in range(num_nodes))
    
    
    remote_base_folder = "/home/tejas/stellar-private" # The base path on the GCP instance
    # local_base_destination = "/home/tejas/work/experiments/stellar-core/" + "collection_"+str(n_collection)+"_rounds_" + str(num_nodes) 
    local_base_destination = "/home/tejas/work/experiments/stellar-core/" + "nr_" + str(num_nodes) + "_test"
    
    # Ensure the local base destination directory exists
    os.makedirs(local_base_destination, exist_ok=True)
    
    
    def copy_folder_from_instance(i):
        """
        Constructs and executes the gcloud compute scp command to copy a specific 
        nodeN folder from instance i to a local folder named after the instance.
        """
        instance_name = f"tsm-sc-{i:03}"
        
        # Calculate the node number (assuming i starts at 0, node starts at 1)
        node_number = i + 1 
        node_folder = f"node{node_number}"
    
        # 1. Define the specific REMOTE source path on the instance
        # Example: /home/tejas/stellar-private/node1
        remote_source_path = os.path.join(remote_base_folder, node_folder)
        
        # 2. Define the LOCAL destination path
        # We'll use the instance name for the subfolder to keep backups separate
        local_destination_path = os.path.join(local_base_destination, instance_name)
        os.makedirs(local_destination_path, exist_ok=True)
        
        # The SCp command requires the remote path to be formatted as:
        # [INSTANCE_NAME]:[REMOTE_SRC]
        remote_source = f"{instance_name}:{remote_source_path}"
        
        # The command reverses the source (remote) and destination (local)
        command = f'gcloud compute scp --zone "{zone}" --project "{project}" \
    --recurse "{remote_source}" "{local_destination_path}"'
    
        print(f"Executing command to copy {node_folder} from {instance_name}: {command}")
        
        # os.system executes the command and returns the exit status (0 for success)
        output = os.system(command)
        
        print(f"Copy from {instance_name} finished with exit code: {output}")
        
        return (instance_name, output)
    
    # ---
    # Execute the copy operation in parallel
    # ---
    
    results = Parallel(n_jobs=20)(
        delayed(copy_folder_from_instance)(i) for i in range(num_nodes)
    )
    
    print("\n--- Summary of Download Results ---")
    print(results)
    
    
    # # Fetch all existing instance names matching tsm-sc-*
    # fetch_cmd = f'''
    # gcloud compute instances list \
    #     --filter="name~'tsm-sc-'" \
    #     --format="value(name)"
    # '''
    # instances_to_delete = subprocess.check_output(fetch_cmd, shell=True).decode().strip().split('\n')
    # instances_to_delete = [name for name in instances_to_delete if name]  # Remove empty entries
    
    # print("\n➡ Existing instances to delete:", instances_to_delete)
    
    # if instances_to_delete:
    #     # Use parallel deletion
    #     def delete_instance(instance_name):
    #         cmd = f'''
    #         gcloud compute instances delete {instance_name} \
    #             --zone={zone} \
    #             --project={project} \
    #             --quiet
    #         '''
    #         print(f"Deleting: {instance_name}")
    #         return subprocess.call(cmd, shell=True)
    
    #     with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    #         futures = [executor.submit(delete_instance, inst) for inst in instances_to_delete]
    #         concurrent.futures.wait(futures)
    
    #     print("🧹 All old tsm-sc-* instances deleted.\n")
    # else:
    #     print("✔ No previous instances found to delete.\n")
    
    
    
    
    # # Fetch all existing instance names matching tsm-sc-*
    # fetch_cmd = f'''
    # gcloud compute instances list \
    #     --filter="name~'tsm-sc-'" \
    #     --format="value(name)"
    # '''
    # instances_to_delete = subprocess.check_output(fetch_cmd, shell=True).decode().strip().split('\n')
    # instances_to_delete = [name for name in instances_to_delete if name]  # Remove empty entries
    
    # print("\n➡ Existing instances to delete:", instances_to_delete)
    
    # if instances_to_delete:
    #     # Use parallel deletion
    #     def delete_instance(instance_name):
    #         cmd = f'''
    #         gcloud compute instances delete {instance_name} \
    #             --zone={zone} \
    #             --project={project} \
    #             --quiet
    #         '''
    #         print(f"Deleting: {instance_name}")
    #         return subprocess.call(cmd, shell=True)
    
    #     with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    #         futures = [executor.submit(delete_instance, inst) for inst in instances_to_delete]
    #         concurrent.futures.wait(futures)
    
    #     print("🧹 All old tsm-sc-* instances deleted.\n")
    # else:
    #     print("✔ No previous instances found to delete.\n")

[main fa79b33] this works for 4 nodes
 4 files changed, 2479 insertions(+), 21 deletions(-)


To github.com:tejas-shivanand-mane/stellar-core.git
   52b2c47..fa79b33  main -> main
From https://github.com/tejas-shivanand-mane/stellar-core
   52b2c47..fa79b33  main       -> origin/main
From https://github.com/tejas-shivanand-mane/stellar-core
   52b2c47..fa79b33  main       -> origin/main
From https://github.com/tejas-shivanand-mane/stellar-core
   52b2c47..fa79b33  main       -> origin/main
From https://github.com/tejas-shivanand-mane/stellar-core
   52b2c47..fa79b33  main       -> origin/main
From https://github.com/tejas-shivanand-mane/stellar-core
   52b2c47..fa79b33  main       -> origin/main
From https://github.com/tejas-shivanand-mane/stellar-core
   52b2c47..fa79b33  main       -> origin/main


Updating 52b2c47..fa79b33
Fast-forward
Updating 52b2c47..fa79b33
Fast-forward
 SetupGCP.ipynb | 2466 +++++++++++++++++++++++++++++++++++++++++++++++++++++++-
 latency.png    |  Bin 86958 -> 83479 bytes
 post.ipynb     |   34 +-
 throughput.png |  Bin 82921 -> 79116 bytes
 4 files changed, 2479 insertions(+), 21 deletions(-)
Updating 52b2c47..fa79b33
Fast-forward
 SetupGCP.ipynb | 2466 +++++++++++++++++++++++++++++++++++++++++++++++++++++++-
 latency.png    |  Bin 86958 -> 83479 bytes
 post.ipynb     |   34 +-
 throughput.png |  Bin 82921 -> 79116 bytes
 4 files changed, 2479 insertions(+), 21 deletions(-)
 SetupGCP.ipynb | 2466 +++++++++++++++++++++++++++++++++++++++++++++++++++++++-
 latency.png    |  Bin 86958 -> 83479 bytes
 post.ipynb     |   34 +-
 throughput.png |  Bin 82921 -> 79116 bytes
 4 files changed, 2479 insertions(+), 21 deletions(-)
Updating 52b2c47..fa79b33
Fast-forward
Updating 52b2c47..fa79b33
Fast-forward
 SetupGCP.ipynb | 2466 ++++++++++++++++++++++++++++++++++++++

From https://github.com/tejas-shivanand-mane/stellar-core
   52b2c47..fa79b33  main       -> origin/main
From https://github.com/tejas-shivanand-mane/stellar-core
   52b2c47..fa79b33  main       -> origin/main


Updating 52b2c47..fa79b33
Fast-forward
Updating 52b2c47..fa79b33
Fast-forward
 SetupGCP.ipynb | 2466 +++++++++++++++++++++++++++++++++++++++++++++++++++++++-
 latency.png    |  Bin 86958 -> 83479 bytes
 post.ipynb     |   34 +-
 throughput.png |  Bin 82921 -> 79116 bytes
 4 files changed, 2479 insertions(+), 21 deletions(-)
 SetupGCP.ipynb | 2466 +++++++++++++++++++++++++++++++++++++++++++++++++++++++-
 latency.png    |  Bin 86958 -> 83479 bytes
 post.ipynb     |   34 +-
 throughput.png |  Bin 82921 -> 79116 bytes
 4 files changed, 2479 insertions(+), 21 deletions(-)
Updating 52b2c47..fa79b33
Fast-forward
 SetupGCP.ipynb | 2466 +++++++++++++++++++++++++++++++++++++++++++++++++++++++-
 latency.png    |  Bin 86958 -> 83479 bytes
 post.ipynb     |   34 +-
 throughput.png |  Bin 82921 -> 79116 bytes
 4 files changed, 2479 insertions(+), 21 deletions(-)
[None, None, None, None, None, None, None, None]
Detected 8 nodes based on tsm_ips.txt.
Cleaning and creating directory for node1. Ports: P

...

Exception ignored in: <function ResourceTracker.__del__ at 0x73af4f78e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-006" --project "ucr-ursa-major-lesani-lab" --command "cd /home/tejas/stellar-private; sudo pkill stellar-core; "
Return code for tsm-sc-006: 256
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-000" --project "ucr-ursa-major-lesani-lab" --command "cd /home/tejas; sudo rm -r stellar-private; "
Return code for tsm-sc-000: 256
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-006" --project "ucr-ursa-major-lesani-lab" --command "cd /home/tejas/stellar-private; nohup /home/tejas/stellar-core/src/stellar-core run --conf node7/stellar-core.cfg > node7/stellar-core.log 2>&1 < /dev/null & disown
"
Return code for tsm-sc-006: 0
Executing command to copy node6 from tsm-sc-005: gcloud compute scp --zone "us-central1-c" --project "ucr-ursa-major-lesani-lab" --recurse "tsm-sc-005:/home/tejas/stellar-private/node6" "/home/tejas/work/experiments/stellar-core/nr_8_test/tsm-sc-005"
Copy from tsm-sc-005 finished with exi

Exception ignored in: <function ResourceTracker.__del__ at 0x71308638e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7122daf8e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ 

Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-007" --project "ucr-ursa-major-lesani-lab" --command "cd /home/tejas/stellar-private; sudo pkill stellar-core; "
Return code for tsm-sc-007: 256
gcloud compute ssh --zone "us-central1-c" "tsm-sc-004" --project "ucr-ursa-major-lesani-lab" --command "cd stellar-core; make -j16; cd; sudo rm -r stellar-private"
256
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-001" --project "ucr-ursa-major-lesani-lab" --command "cd /home/tejas/stellar-private; nohup /home/tejas/stellar-core/src/stellar-core run --conf node2/stellar-core.cfg > node2/stellar-core.log 2>&1 < /dev/null & disown
"
Return code for tsm-sc-001: 0
Executing command to copy node7 from tsm-sc-006: gcloud compute scp --zone "us-central1-c" --project "ucr-ursa-major-lesani-lab" --recurse "tsm-sc-006:/home/tejas/stellar-private/node7" "/home/tejas/work/experiments/stellar-core/nr_8_test/tsm-sc-006"
Copy from tsm-sc-006 finished with exit code: 0
gcloud compute 

Exception ignored in: <function ResourceTracker.__del__ at 0x7b5f59586020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
